# KPI Analysis V1 - Budget App Premium

This notebook moves from dataset validation to a first business-oriented KPI analysis for a premium personal budgeting app.

Business questions covered here:
- How many new users join the product over time?
- How effective is activation overall and by acquisition channel?
- How strong is trial-to-paid conversion overall and by channel?
- How much does activation improve conversion?
- What is the mix between monthly and annual subscriptions?
- What do revenue and ARPU look like at a first level?
- What do early usage and engagement patterns suggest?

Scope limits for this V1:
- no detailed cohort analysis yet
- no predictive churn modelling
- some usage-based comparisons remain directional because `monthly_customer_activity` does not materialize explicit zero-activity months

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

DATA_DIR = Path('../data/raw')
DATASET_END = pd.Timestamp('2025-12-31')
MONTHS = pd.date_range('2025-01-01', '2025-12-01', freq='MS')

## 2. Load Data

In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
subscriptions = pd.read_csv(
    DATA_DIR / 'subscriptions.csv',
    parse_dates=['trial_start_date', 'trial_end_date', 'subscription_start_date', 'subscription_end_date'],
)
payments = pd.read_csv(DATA_DIR / 'payments.csv', parse_dates=['payment_date'])
product_events = pd.read_csv(DATA_DIR / 'product_events.csv', parse_dates=['event_date'])
monthly_customer_activity = pd.read_csv(
    DATA_DIR / 'monthly_customer_activity.csv',
    parse_dates=['activity_month'],
)

print(f'customers: {customers.shape[0]:,}')
print(f'subscriptions: {subscriptions.shape[0]:,}')
print(f'payments: {payments.shape[0]:,}')
print(f'product_events: {product_events.shape[0]:,}')
print(f'monthly_customer_activity: {monthly_customer_activity.shape[0]:,}')

## 3. KPI Base Tables

In [ ]:
activation_events = product_events.merge(customers[['customer_id', 'signup_date']], on='customer_id', how='left')
activation_events = activation_events[
    activation_events['event_date'] <= activation_events['signup_date'] + pd.Timedelta(days=6)
]

activation_status = (
    activation_events.groupby('customer_id')['event_type']
    .agg(list)
    .apply(
        lambda events: (
            any(event in {'bank_account_connected', 'transaction_imported'} for event in events)
            and 'budget_created' in events
        )
    )
    .rename('is_activated')
)

customers_kpi = customers.merge(activation_status, on='customer_id', how='left')
customers_kpi['is_activated'] = customers_kpi['is_activated'].eq(True)
customers_kpi['signup_month'] = customers_kpi['signup_date'].values.astype('datetime64[M]')

subscriptions_kpi = subscriptions.merge(
    customers_kpi[['customer_id', 'acquisition_channel', 'is_activated', 'signup_month']],
    on='customer_id',
    how='left',
)

paid_customers = subscriptions_kpi[subscriptions_kpi['converted_to_paid']].copy()
paid_payments = payments[payments['payment_status'] == 'paid'].copy()

latest_usage = (
    monthly_customer_activity.sort_values(['customer_id', 'activity_month'])
    .groupby('customer_id')
    .tail(1)[['customer_id', 'usage_segment', 'activity_month']]
    .rename(columns={'activity_month': 'latest_activity_month'})
)

paid_customers = paid_customers.merge(latest_usage, on='customer_id', how='left')
paid_customers['usage_segment'] = paid_customers['usage_segment'].fillna('no_observed_activity')

## 4. Acquisition and New User Volume

In [ ]:
new_users_monthly = (
    customers_kpi.groupby('signup_month')['customer_id']
    .count()
    .reindex(MONTHS, fill_value=0)
    .rename('new_users')
    .to_frame()
)

new_users_monthly

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
new_users_monthly['new_users'].plot(ax=ax, marker='o', color='#2563eb')
ax.set_title('New Users by Month')
ax.set_xlabel('Signup month')
ax.set_ylabel('New users')
plt.tight_layout()
plt.show()

**Observation**

The user inflow is stable enough to support monthly KPI analysis. The dataset does not simulate sharp seasonality, which is acceptable for a first MVP business case.

## 5. Activation

In [ ]:
activation_global = customers_kpi['is_activated'].mean()
activation_by_channel = (
    customers_kpi.groupby('acquisition_channel')['is_activated']
    .mean()
    .sort_values(ascending=False)
    .rename('activation_rate')
    .to_frame()
)

pd.DataFrame([
    {'kpi': 'Activation rate', 'value': activation_global}
])

In [ ]:
activation_by_channel

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
activation_by_channel['activation_rate'].sort_values().plot(kind='barh', ax=ax, color='#16a34a')
ax.set_title('Activation Rate by Acquisition Channel')
ax.set_xlabel('Activation rate')
plt.tight_layout()
plt.show()

**Observation**

Activation is strong overall. Organic and Referral channels should be treated as higher-quality top-of-funnel sources, while Paid Social appears weaker on early product adoption.

## 6. Trial to Paid Conversion

In [ ]:
conversion_global = subscriptions_kpi['converted_to_paid'].mean()
conversion_by_channel = (
    subscriptions_kpi.groupby('acquisition_channel')['converted_to_paid']
    .mean()
    .sort_values(ascending=False)
    .rename('conversion_rate')
    .to_frame()
)
conversion_by_activation = (
    subscriptions_kpi.groupby('is_activated')['converted_to_paid']
    .mean()
    .rename('conversion_rate')
    .to_frame()
)

pd.DataFrame([
    {'kpi': 'Trial to paid conversion rate', 'value': conversion_global}
])

In [ ]:
display(conversion_by_channel)
display(conversion_by_activation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

conversion_by_channel['conversion_rate'].sort_values().plot(kind='barh', ax=axes[0], color='#3b82f6')
axes[0].set_title('Conversion Rate by Acquisition Channel')
axes[0].set_xlabel('Conversion rate')

conversion_by_activation['conversion_rate'].rename(index={False: 'Not activated', True: 'Activated'}).plot(
    kind='bar', ax=axes[1], color=['#f97316', '#16a34a']
)
axes[1].set_title('Conversion Rate: Activated vs Not Activated')
axes[1].set_xlabel('Activation status')
axes[1].set_ylabel('Conversion rate')

plt.tight_layout()
plt.show()

**Observation**

Conversion is materially stronger for activated users. This confirms that onboarding quality is one of the first business levers to improve paid growth.

## 7. Plan Mix

In [ ]:
plan_mix = (
    paid_customers['plan_type']
    .value_counts(normalize=True)
    .rename('share')
    .to_frame()
)
plan_mix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plan_mix['share'].plot(kind='bar', ax=ax, color=['#dc2626', '#0f766e'])
ax.set_title('Plan Mix Among Paid Customers')
ax.set_xlabel('Plan type')
ax.set_ylabel('Share')
plt.tight_layout()
plt.show()

**Observation**

Monthly plans remain the dominant offer, which is expected in a B2C subscription product. The annual share is still large enough to matter for revenue stability.

## 8. Revenue and ARPU

In [ ]:
arpu_simple = paid_payments['amount'].sum() / paid_customers['customer_id'].nunique()

mrr_monthly_rows = []
for month_start in MONTHS:
    month_end = month_start + pd.offsets.MonthEnd(1)
    active_paid = subscriptions_kpi[
        subscriptions_kpi['converted_to_paid']
        & (subscriptions_kpi['subscription_start_date'] <= month_end)
        & (
            subscriptions_kpi['subscription_end_date'].isna()
            | (subscriptions_kpi['subscription_end_date'] > month_end)
        )
    ]
    mrr_monthly_rows.append(
        {
            'month': month_start,
            'active_paid_customers': active_paid['customer_id'].nunique(),
            'approx_mrr': active_paid['monthly_price'].sum(),
        }
    )

mrr_monthly = pd.DataFrame(mrr_monthly_rows)

revenue_summary = pd.DataFrame([
    {'kpi': 'Simple ARPU', 'value': arpu_simple},
    {'kpi': 'Approximate end-of-period MRR', 'value': mrr_monthly.iloc[-1]['approx_mrr']},
    {'kpi': 'End-of-period active paid customers', 'value': mrr_monthly.iloc[-1]['active_paid_customers']},
])
revenue_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
mrr_monthly.plot(x='month', y='approx_mrr', marker='o', ax=ax, color='#7c3aed')
ax.set_title('Approximate Monthly MRR Evolution')
ax.set_xlabel('Month')
ax.set_ylabel('Approximate MRR')
plt.tight_layout()
plt.show()

**Observation**

The MRR line gives a simple but useful business view of subscription momentum. In this V1, MRR is approximated from active paid subscriptions and normalized monthly prices.

## 9. Usage and Basic Engagement

Important note:
- `monthly_customer_activity` contains observed months with events only
- it does not materialize explicit zero-activity months
- usage comparisons below are therefore useful for directional interpretation, not for final retention logic

In [ ]:
engagement_summary = pd.Series({
    'avg_monthly_app_opens': monthly_customer_activity['nb_app_opens'].mean(),
    'avg_monthly_transactions_imported': monthly_customer_activity['nb_transactions_imported'].mean(),
    'avg_monthly_dashboard_views': monthly_customer_activity['nb_dashboard_views'].mean(),
})
engagement_summary.to_frame('value')

In [ ]:
usage_distribution_paid = (
    paid_customers['usage_segment']
    .value_counts(normalize=True)
    .rename('share')
    .to_frame()
)

plan_by_usage = (
    paid_customers.groupby('usage_segment')['plan_type']
    .value_counts(normalize=True)
    .rename('share')
    .reset_index()
)

display(usage_distribution_paid)
display(plan_by_usage)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

usage_distribution_paid['share'].sort_values().plot(kind='barh', ax=axes[0], color='#8b5cf6')
axes[0].set_title('Latest Observed Usage Segment Among Paid Customers')
axes[0].set_xlabel('Share')

plan_by_usage.pivot(index='usage_segment', columns='plan_type', values='share').fillna(0).plot(
    kind='bar', stacked=True, ax=axes[1], color=['#ef4444', '#14b8a6']
)
axes[1].set_title('Plan Mix by Latest Observed Usage Segment')
axes[1].set_xlabel('Usage segment')
axes[1].set_ylabel('Share')

plt.tight_layout()
plt.show()

**Observation**

Usage segmentation is already useful for framing business conversations around engagement. It should still be treated as exploratory until a more rigorous activity calendar is built.

## 10. First Comparisons by Channel and Segment

In [ ]:
channel_summary = (
    subscriptions_kpi.groupby('acquisition_channel')
    .agg(
        customers=('customer_id', 'count'),
        activation_rate=('is_activated', 'mean'),
        conversion_rate=('converted_to_paid', 'mean'),
    )
    .sort_values('conversion_rate', ascending=False)
)

paid_revenue_by_channel = (
    paid_customers.groupby('acquisition_channel')
    .agg(
        paid_customers=('customer_id', 'nunique'),
        approx_end_mrr=('monthly_price', 'sum'),
    )
)

channel_summary = channel_summary.join(paid_revenue_by_channel, how='left').fillna(0)
channel_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

channel_summary['conversion_rate'].sort_values().plot(kind='barh', ax=axes[0], color='#2563eb')
axes[0].set_title('Conversion Rate by Acquisition Channel')
axes[0].set_xlabel('Conversion rate')

channel_summary['approx_end_mrr'].sort_values().plot(kind='barh', ax=axes[1], color='#9333ea')
axes[1].set_title('Approximate End-of-Period MRR by Channel')
axes[1].set_xlabel('Approximate MRR')

plt.tight_layout()
plt.show()

**Observation**

Acquisition channels should not be judged on volume alone. Organic and Referral look stronger on both quality and monetization, while Paid Social brings weaker conversion quality.

## 11. Final Takeaways

In [ ]:
activation_rate = customers_kpi['is_activated'].mean()
conversion_rate = subscriptions_kpi['converted_to_paid'].mean()
annual_share = paid_customers['plan_type'].eq('annual').mean()
best_channel = conversion_by_channel.index[0]
worst_channel = conversion_by_channel.index[-1]

print('Main business takeaways')
print(f'- The product acquires {len(customers_kpi):,} users over the period with a steady monthly inflow.')
print(f'- Activation is solid at {activation_rate:.1%}, which supports downstream monetization.')
print(f'- Trial-to-paid conversion reaches {conversion_rate:.1%}, with {best_channel} outperforming and {worst_channel} underperforming.')
print('- Activated users convert materially better than non-activated users, confirming onboarding as a core lever.')
print(f'- Annual plans represent {annual_share:.1%} of paid customers and help diversify revenue beyond monthly subscriptions.')
print('- Revenue and engagement metrics are strong enough to justify a next notebook focused on churn, retention and cohort analysis.')